In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
df = spark.read.load("/Volumes/pl_big_6/bronze/match_data")
# .where(F.to_date(F.col('created_at'))>=F.current_date())
# df_transfer_data = spark.read.load("/Volumes/pl_big_6/bronze/transfer_data").where(F.to_date(F.col('created_at'))>=F.current_date())
#Date,HomeTeam,AwayTeam -- match_data
#season,league,club,player_name -- transfer_data

In [0]:
df_match_data = df.select(F.coalesce(F.try_to_date(F.col("Date"),'dd/MM/yyyy'),F.try_to_date(F.col("Date"),'dd/MM/yy')).alias('Date'),
    F.col("HomeTeam"),
    F.col("AwayTeam"),
    F.col("FTHG").cast('int'),
    F.col("FTAG").cast('int'),
    F.col("FTR"),
    F.col('created_at'))

In [0]:
df_match_data.display()

In [0]:
df_match_data_season = df_match_data.select(
    F.when(F.month(F.col("Date"))>=7,F.concat(F.year(F.col("Date")),F.lit('-'),F.year(F.col("Date"))+1)).otherwise(F.concat(F.year(F.col("Date"))-1,F.lit('-'),F.year(F.col("Date")))).alias('season'),'*'
)

In [0]:
df_match_data_season.display()


In [0]:
# # create emppty silver tables
# dbutils.fs.rm("/Volumes/pl_big_6/silver/pl_data/match_data",recurse = True)
# schema = df_match_data_season.schema

# empty_df = spark.createDataFrame([], schema)

# empty_df.write.format("delta").option("enableChangeDataFeed", "true").save("/Volumes/pl_big_6/silver/pl_data/match_data")

In [0]:
# %sql
# ALTER TABLE delta.`/Volumes/pl_big_6/silver/pl_data/match_data`
# SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
silver_match_data = DeltaTable.forPath(spark, "/Volumes/pl_big_6/silver/pl_data/match_data")

silver_match_data.alias("t").merge(
    df_match_data_season.alias("s"),
    "t.Date = s.Date and t.HomeTeam = s.HomeTeam and t.AwayTeam = s.AwayTeam"
).whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()